In [1]:

# Cell 1 — Load modeling data

from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px

from pricewise.data.features import (
    FEATURE_COLUMNS,
    TARGET_COLUMN,
    build_feature_table,
)

# Works when Jupyter starts from the project root
# or from inside the notebooks directory.
ROOT = Path.cwd().resolve()

if ROOT.name == "notebooks":
    ROOT = ROOT.parent

DATA_PATH = (
    ROOT / "data/processed/cereal_focus.parquet"
)

assert DATA_PATH.is_file(), DATA_PATH

# Our two-product sales dataset
sales = pd.read_parquet(DATA_PATH)

# The reusable feature engineering pipeline
data = build_feature_table(sales)

assert len(data) == 53_647

print("PRICEWISE | DEMAND MODELING")
print("=" * 40)

print(f"Original rows: {len(sales):,}")
print(f"Model rows:    {len(data):,}")
print(f"Features:      {len(FEATURE_COLUMNS)}")

display(data[FEATURE_COLUMNS + [TARGET_COLUMN]].head())


PRICEWISE | DEMAND MODELING
Original rows: 59,466
Model rows:    53,647
Features:      10


,UPC,STORE,CANDIDATE_PRICE,LAG_1_PRICE,LAG_1_UNITS,AVG_UNITS_PREV_4W,PRICE_CHANGE_PCT,WEEK_SIN,WEEK_COS,PROMO_CONTEXT,MOVE
4,1600066610,2,2.9,2.9,90.0,80.50,0.0,0.464723,0.885456,UNMARKED,60
5,1600066610,2,2.9,2.9,60.0,74.50,0.0,0.568065,0.822984,UNMARKED,40
6,1600066610,2,2.9,2.9,40.0,68.50,0.0,0.663123,0.748511,UNMARKED,54
7,1600066610,2,2.9,2.9,54.0,61.00,0.0,0.748511,0.663123,UNMARKED,67
8,1600066610,2,2.9,2.9,67.0,55.25,0.0,0.822984,0.568065,UNMARKED,41


In [2]:

# Cell 2 — Chronological split

train_df = data.loc[
    data["WEEK"] <= 257
].copy()

val_df = data.loc[
    data["WEEK"].between(258, 342)
].copy()

test_df = data.loc[
    data["WEEK"] >= 343
].copy()


def make_features(frame):
    X = frame[FEATURE_COLUMNS].copy()

    # Treat identifiers as categories, not quantities.
    for column in ["UPC", "STORE", "PROMO_CONTEXT"]:
        X[column] = X[column].astype(str)

    return X


X_train = make_features(train_df)
y_train = train_df[TARGET_COLUMN].copy()

X_val = make_features(val_df)
y_val = val_df[TARGET_COLUMN].copy()

X_test = make_features(test_df)
y_test = test_df[TARGET_COLUMN].copy()


assert len(train_df) == 37_912
assert len(val_df) == 7_578
assert len(test_df) == 8_157

assert X_train.notna().all().all()
assert X_val.notna().all().all()

print("TRAINING:", X_train.shape, y_train.shape)
print("VALIDATION:", X_val.shape, y_val.shape)
print("TESTING:", X_test.shape, y_test.shape)


TRAINING: (37912, 10) (37912,)
VALIDATION: (7578, 10) (7578,)
TESTING: (8157, 10) (8157,)


In [3]:

# Cell 3 — Evaluation and naive forecasts

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
)


def evaluate(name, actual, predicted):

    actual = np.asarray(actual, dtype=float)
    predicted = np.asarray(predicted, dtype=float)

    mae = mean_absolute_error(actual, predicted)

    rmse = np.sqrt(
        mean_squared_error(actual, predicted)
    )

    wape = (
        100
        * np.abs(actual - predicted).sum()
        / actual.sum()
    )

    return {
        "Model": name,
        "MAE": mae,
        "RMSE": rmse,
        "WAPE (%)": wape,
    }


# Baseline A: Same units as previous week
last_week_pred = val_df["LAG_1_UNITS"].to_numpy()

# Baseline B: Average units over last four weeks
rolling_pred = (
    val_df["AVG_UNITS_PREV_4W"].to_numpy()
)

results = [
    evaluate(
        "Last Week",
        y_val,
        last_week_pred,
    ),
    evaluate(
        "4-Week Average",
        y_val,
        rolling_pred,
    ),
]

display(pd.DataFrame(results).round(2))


,Model,MAE,RMSE,WAPE (%)
0,Last Week,49.50,135.0,92.31
1,4-Week Average,48.44,110.0,90.35


In [4]:

# Cell 4 — Random Forest demand estimator

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor


CATEGORICAL = [
    "UPC",
    "STORE",
    "PROMO_CONTEXT",
]

NUMERIC = [
    column
    for column in FEATURE_COLUMNS
    if column not in CATEGORICAL
]


# Step A: Prepare model inputs
preprocessor = ColumnTransformer(
    transformers=[
        (
            "categories",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False,
            ),
            CATEGORICAL,
        ),
        (
            "numbers",
            "passthrough",
            NUMERIC,
        ),
    ],
    remainder="drop",
)


# Step B: Define the model
forest = RandomForestRegressor(
    n_estimators=100,
    max_depth=14,
    min_samples_leaf=5,
    random_state=42,
    n_jobs=-1,
)


# Step C: Combine preprocessing and model
demand_model = Pipeline([
    ("preprocess", preprocessor),
    ("model", forest),
])


# Step D: Train
demand_model.fit(X_train, y_train)


# Step E: Validate
val_predictions = np.maximum(
    demand_model.predict(X_val),
    0,
)


# Step F: Compare against baselines
results.append(
    evaluate(
        "Random Forest",
        y_val,
        val_predictions,
    )
)

comparison = (
    pd.DataFrame(results)
    .sort_values("MAE")
    .reset_index(drop=True)
)

display(comparison.round(2))


,Model,MAE,RMSE,WAPE (%)
0,Random Forest,27.30,80.4,50.91
1,4-Week Average,48.44,110.0,90.35
2,Last Week,49.50,135.0,92.31


In [5]:

# Cell 5 — Validation performance

fig = px.bar(
    comparison,
    x="Model",
    y="MAE",
    text="MAE",
    title="PriceWise | Validation Forecast Error",
    labels={
        "MAE": "Mean Absolute Error (Units)",
    },
)

fig.update_traces(
    texttemplate="%{text:.2f}",
    textposition="outside",
)

fig.update_layout(
    template="plotly_white",
    height=440,
    yaxis_range=[
        0,
        comparison["MAE"].max() * 1.2,
    ],
)

fig.show()


# Inspect errors for our two products separately
product_results = []

for upc, name in {
    "1600066610": "Cheerios 15 oz",
    "3800000120": "Corn Flakes 18 oz",
}.items():

    mask = val_df["UPC"].eq(upc).to_numpy()

    metrics = evaluate(
        name,
        y_val.to_numpy()[mask],
        val_predictions[mask],
    )

    product_results.append(metrics)

display(
    pd.DataFrame(product_results).round(2)
)


,Model,MAE,RMSE,WAPE (%)
0,Cheerios 15 oz,32.72,97.87,46.70
1,Corn Flakes 18 oz,21.89,57.95,58.83


In [6]:

# Cell 6 — Model price-response diagnostic

TARGET_UPC = "1600066610"
TARGET_STORE = 101

# Select one historical validation scenario.
selected = (
    val_df.loc[
        (val_df["UPC"] == TARGET_UPC)
        & (val_df["STORE"] == TARGET_STORE)
    ]
    .sort_values("WEEK")
    .iloc[0]
)

# Price support learned only from training history.
historical = train_df.loc[
    (train_df["UPC"] == TARGET_UPC)
    & (train_df["STORE"] == TARGET_STORE),
    "UNIT_PRICE",
]

assert not historical.empty

reference_price = float(selected["LAG_1_PRICE"])

price_min = max(
    float(historical.quantile(0.10)),
    reference_price * 0.90,
)

price_max = min(
    float(historical.quantile(0.90)),
    reference_price * 1.10,
)

if price_min > price_max:
    raise ValueError(
        "No feasible candidate price interval"
    )

# Generate feasible candidate prices.
prices = np.unique(
    np.round(
        np.linspace(price_min, price_max, 25),
        2,
    )
)

# Copy the exact same historical context
# for every proposed price.
scenarios = pd.concat(
    [X_val.loc[[selected.name]]] * len(prices),
    ignore_index=True,
)

scenarios["CANDIDATE_PRICE"] = prices

# Keep derived price features consistent.
scenarios["PRICE_CHANGE_PCT"] = (
    100
    * (
        scenarios["CANDIDATE_PRICE"]
        / scenarios["LAG_1_PRICE"]
        - 1
    )
)

# Predict demand for each candidate price.
predicted_units = np.maximum(
    demand_model.predict(scenarios),
    0,
)

response = pd.DataFrame({
    "price": prices,
    "predicted_units": predicted_units,
})

print("Forecast week:", int(selected["WEEK"]))
print("Product:", TARGET_UPC)
print("Store:", TARGET_STORE)
print("Reference price:", reference_price)

display(response.head())

fig = px.line(
    response,
    x="price",
    y="predicted_units",
    markers=True,
    title="PriceWise | Predicted Demand vs Candidate Price",
    labels={
        "price": "Candidate Unit Price ($)",
        "predicted_units": "Predicted Weekly Units",
    },
)

fig.update_layout(
    template="plotly_white",
    height=500,
)

fig.show()

# Check the direction of predicted price response.
steps = np.diff(predicted_units)

print(
    "Price increases accompanied by "
    "higher predicted demand:",
    int((steps > 1e-8).sum()),
    "of",
    len(steps),
)

print(
    "Forecast demand range:",
    round(float(predicted_units.max()
                - predicted_units.min()), 2),
    "units",
)


Forecast week: 258
Product: 1600066610
Store: 101
Reference price: 3.22


,price,predicted_units
0,2.90,83.357203
1,2.93,77.263783
2,2.95,77.263783
3,2.98,76.343843
4,3.01,56.902436


Price increases accompanied by higher predicted demand: 7 of 24
Forecast demand range: 48.63 units


In [7]:

# Cell 7 — Train monotonic gradient boosting

from sklearn.base import clone
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.pipeline import Pipeline

# Reuse the preprocessor fitted on training data
# to identify the transformed feature positions.
fitted_preprocessor = (
    demand_model.named_steps["preprocess"]
)

feature_names = list(
    fitted_preprocessor.get_feature_names_out()
)

# -1 = monotonic decreasing
#  0 = no monotonic restriction
monotonic_constraints = [
    -1 if name in {
        "numbers__CANDIDATE_PRICE",
        "numbers__PRICE_CHANGE_PCT",
    }
    else 0
    for name in feature_names
]

assert monotonic_constraints.count(-1) == 2

print("CONSTRAINED FEATURES")

for name, constraint in zip(
    feature_names,
    monotonic_constraints
):
    if constraint != 0:
        print(name, constraint)

# Define a gradient-boosting estimator.
boosting = HistGradientBoostingRegressor(
    loss="squared_error",
    learning_rate=0.06,
    max_iter=160,
    max_leaf_nodes=20,
    min_samples_leaf=25,
    l2_regularization=2.0,
    monotonic_cst=monotonic_constraints,
    early_stopping=False,
    random_state=42,
)

# Clone prevents sharing fitted preprocessing state
# with our original Random Forest pipeline.
monotone_model = Pipeline([
    (
        "preprocess",
        clone(fitted_preprocessor),
    ),
    (
        "model",
        boosting,
    ),
])

# Train using only the training period.
monotone_model.fit(X_train, y_train)

# Predict validation demand.
monotone_predictions = np.maximum(
    monotone_model.predict(X_val),
    0,
)

monotone_metrics = evaluate(
    "Monotonic Boosting",
    y_val,
    monotone_predictions,
)

# Compare against our previous benchmarks.
updated_comparison = (
    pd.concat(
        [
            comparison,
            pd.DataFrame([monotone_metrics]),
        ],
        ignore_index=True,
    )
    .sort_values("MAE")
    .reset_index(drop=True)
)

display(updated_comparison.round(2))


CONSTRAINED FEATURES
numbers__CANDIDATE_PRICE -1
numbers__PRICE_CHANGE_PCT -1


,Model,MAE,RMSE,WAPE (%)
0,Random Forest,27.30,80.4,50.91
1,Monotonic Boosting,33.99,89.2,63.39
2,4-Week Average,48.44,110.0,90.35
3,Last Week,49.50,135.0,92.31


In [8]:

# Cell 8 — Compare per-product validation errors

product_names = {
    "1600066610": "Cheerios 15 oz",
    "3800000120": "Corn Flakes 18 oz",
}

rows = []

for upc, product_name in product_names.items():

    mask = val_df["UPC"].eq(upc).to_numpy()

    actual = y_val.to_numpy()[mask]

    for model_name, predictions in [
        ("Random Forest", val_predictions),
        ("Monotonic Boosting", monotone_predictions),
    ]:

        metrics = evaluate(
            model_name,
            actual,
            predictions[mask],
        )

        rows.append({
            "Product": product_name,
            **metrics,
        })

product_comparison = pd.DataFrame(rows)

display(product_comparison.round(2))

fig = px.bar(
    product_comparison,
    x="Product",
    y="MAE",
    color="Model",
    barmode="group",
    text="MAE",
    title="PriceWise | Validation Error by Product",
)

fig.update_traces(
    texttemplate="%{text:.1f}",
    textposition="outside",
)

fig.update_layout(
    template="plotly_white",
    height=480,
)

fig.show()


,Product,Model,MAE,RMSE,WAPE (%)
0,Cheerios 15 oz,Random Forest,32.72,97.87,46.70
1,Cheerios 15 oz,Monotonic Boosting,38.29,105.65,54.64
2,Corn Flakes 18 oz,Random Forest,21.89,57.95,58.83
3,Corn Flakes 18 oz,Monotonic Boosting,29.71,68.99,79.83


In [9]:

# Cell 9 — Random Forest vs constrained boosting

constrained_units = np.maximum(
    monotone_model.predict(scenarios),
    0,
)

curve_comparison = pd.DataFrame({
    "price": prices,
    "Random Forest": predicted_units,
    "Monotonic Boosting": constrained_units,
})

# Monotonicity check:
# increasing price must not increase predicted demand.
price_steps = np.diff(constrained_units)

violations = int((price_steps > 1e-6).sum())

print("PRICE RESPONSE DIAGNOSTICS")
print("=" * 40)

print("Constrained model violations:", violations)

print(
    "Random Forest demand range:",
    round(float(np.ptp(predicted_units)), 2),
)

print(
    "Monotonic model demand range:",
    round(float(np.ptp(constrained_units)), 2),
)

assert violations == 0, (
    "Constrained model violates price monotonicity"
)

# Convert to long format for plotting.
plot_data = curve_comparison.melt(
    id_vars="price",
    var_name="Model",
    value_name="Predicted Units",
)

fig = px.line(
    plot_data,
    x="price",
    y="Predicted Units",
    color="Model",
    markers=True,
    title="PriceWise | Comparing Demand Responses",
    labels={
        "price": "Candidate Unit Price ($)",
        "Predicted Units": "Predicted Weekly Units",
    },
)

fig.update_layout(
    template="plotly_white",
    height=540,
    hovermode="x unified",
)

fig.show()


PRICE RESPONSE DIAGNOSTICS
Constrained model violations: 0
Random Forest demand range: 48.63
Monotonic model demand range: 81.42


In [10]:

# Cell 10 — Estimate price elasticity from training data

import statsmodels.formula.api as smf

CHEERIOS_UPC = "1600066610"
CORN_FLAKES_UPC = "3800000120"

# Never use validation/test data to estimate elasticity.
elasticity_data = train_df.copy()

elasticity_data["UPC"] = (
    elasticity_data["UPC"].astype(str)
)

elasticity_data["PROMO_CONTEXT"] = (
    elasticity_data["PROMO_CONTEXT"].astype(str)
)

# Product-store fixed-effect identifier.
elasticity_data["STORE_SKU"] = (
    elasticity_data["UPC"]
    + "_"
    + elasticity_data["STORE"].astype(str)
)

# Log transformations
elasticity_data["LOG_UNITS"] = np.log(
    elasticity_data["MOVE"]
)

elasticity_data["LOG_PRICE"] = np.log(
    elasticity_data["UNIT_PRICE"]
)

elasticity_data["LOG_PREV_UNITS"] = np.log1p(
    elasticity_data["LAG_1_UNITS"]
)

# Separate price coefficients for the two products.
elasticity_data["LOG_PRICE_CHEERIOS"] = np.where(
    elasticity_data["UPC"] == CHEERIOS_UPC,
    elasticity_data["LOG_PRICE"],
    0.0,
)

elasticity_data["LOG_PRICE_CORN"] = np.where(
    elasticity_data["UPC"] == CORN_FLAKES_UPC,
    elasticity_data["LOG_PRICE"],
    0.0,
)

# Fixed effects help control persistent store-product
# differences. Other controls partially account for
# promotions, seasonality, time, and recent sales.
formula = """
LOG_UNITS ~
    LOG_PRICE_CHEERIOS
    + LOG_PRICE_CORN
    + C(STORE_SKU)
    + C(PROMO_CONTEXT)
    + WEEK_SIN
    + WEEK_COS
    + WEEK
    + LOG_PREV_UNITS
"""

elasticity_model = smf.ols(
    formula=formula,
    data=elasticity_data,
).fit(
    cov_type="cluster",
    cov_kwds={
        "groups": elasticity_data["STORE_SKU"]
    },
)

coefficient_names = {
    "LOG_PRICE_CHEERIOS": "Cheerios 15 oz",
    "LOG_PRICE_CORN": "Corn Flakes 18 oz",
}

confidence_intervals = elasticity_model.conf_int()

elasticity_results = pd.DataFrame([
    {
        "Product": product_name,
        "Elasticity": elasticity_model.params[coefficient],
        "CI lower": confidence_intervals.loc[coefficient, 0],
        "CI upper": confidence_intervals.loc[coefficient, 1],
    }
    for coefficient, product_name
    in coefficient_names.items()
])

print("PRICEWISE | ELASTICITY ESTIMATION")
print("=" * 40)

display(elasticity_results.round(3))


PRICEWISE | ELASTICITY ESTIMATION


,Product,Elasticity,CI lower,CI upper
0,Cheerios 15 oz,-1.864,-1.999,-1.729
1,Corn Flakes 18 oz,-2.212,-2.317,-2.106


In [11]:

# Cell 11 — Hybrid demand estimator

estimated_elasticities = {
    CHEERIOS_UPC: float(
        elasticity_model.params["LOG_PRICE_CHEERIOS"]
    ),
    CORN_FLAKES_UPC: float(
        elasticity_model.params["LOG_PRICE_CORN"]
    ),
}


def hybrid_predict(X):
    """
    Estimate demand at proposed prices.

    Reference demand comes from Random Forest.
    Price adjustment uses estimated elasticity.
    """

    X = X.copy()

    # -----------------------------------
    # A. Reference-price prediction
    # -----------------------------------

    reference = X.copy()

    reference["CANDIDATE_PRICE"] = (
        reference["LAG_1_PRICE"]
    )

    reference["PRICE_CHANGE_PCT"] = 0.0

    reference_demand = np.maximum(
        demand_model.predict(reference),
        0,
    )

    # -----------------------------------
    # B. Candidate/reference price ratio
    # -----------------------------------

    price_ratio = (
        X["CANDIDATE_PRICE"].to_numpy(dtype=float)
        / X["LAG_1_PRICE"].to_numpy(dtype=float)
    )

    if not np.isfinite(price_ratio).all():
        raise ValueError("Invalid price ratios")

    if (price_ratio <= 0).any():
        raise ValueError("Prices must be positive")

    # -----------------------------------
    # C. Match product to elasticity
    # -----------------------------------

    elasticity = (
        X["UPC"]
        .astype(str)
        .map(estimated_elasticities)
        .to_numpy(dtype=float)
    )

    if not np.isfinite(elasticity).all():
        raise ValueError(
            "Missing or invalid product elasticity"
        )

    # -----------------------------------
    # D. Adjust demand for candidate price
    # -----------------------------------

    return (
        reference_demand
        * np.power(price_ratio, elasticity)
    )


# Validation-period predictions
hybrid_predictions = hybrid_predict(X_val)

hybrid_metrics = evaluate(
    "Hybrid Demand",
    y_val,
    hybrid_predictions,
)

# Compare against all previous approaches.
all_model_results = pd.concat(
    [
        updated_comparison,
        pd.DataFrame([hybrid_metrics]),
    ],
    ignore_index=True,
).sort_values("MAE").reset_index(drop=True)

display(all_model_results.round(2))

# Check how well the Random Forest predicts
# at the reference price, before adjustment.
reference_features = X_val.copy()
reference_features["CANDIDATE_PRICE"] = (
    reference_features["LAG_1_PRICE"]
)
reference_features["PRICE_CHANGE_PCT"] = 0.0

reference_predictions = np.maximum(
    demand_model.predict(reference_features),
    0,
)

display(pd.DataFrame([
    evaluate(
        "RF at Reference Price",
        y_val,
        reference_predictions,
    )
]).round(2))


,Model,MAE,RMSE,WAPE (%)
0,Random Forest,27.30,80.40,50.91
1,Monotonic Boosting,33.99,89.20,63.39
2,Hybrid Demand,44.69,159.03,83.35
3,4-Week Average,48.44,110.00,90.35
4,Last Week,49.50,135.00,92.31


,Model,MAE,RMSE,WAPE (%)
0,RF at Reference Price,33.08,93.48,61.7


In [12]:

# Cell 12 — Compare all three price-response models

hybrid_units = hybrid_predict(scenarios)

curves = pd.DataFrame({
    "Candidate Price": prices,
    "Random Forest": predicted_units,
    "Monotonic Boosting": constrained_units,
    "Hybrid Demand": hybrid_units,
})

long_curves = curves.melt(
    id_vars="Candidate Price",
    var_name="Model",
    value_name="Predicted Units",
)

fig = px.line(
    long_curves,
    x="Candidate Price",
    y="Predicted Units",
    color="Model",
    markers=True,
    title="PriceWise | Demand Response Comparison",
    labels={
        "Candidate Price": "Candidate Unit Price ($)",
        "Predicted Units": "Predicted Weekly Units",
    },
)

fig.update_layout(
    template="plotly_white",
    height=550,
    hovermode="x unified",
)

fig.show()

# Price-response diagnostics
hybrid_violations = int(
    (np.diff(hybrid_units) > 1e-8).sum()
)

print("\nPRICEWISE | HYBRID DIAGNOSTICS")
print("=" * 40)

print(
    "Estimated Cheerios elasticity:",
    round(estimated_elasticities[CHEERIOS_UPC], 3)
)

print(
    "Estimated Corn Flakes elasticity:",
    round(estimated_elasticities[CORN_FLAKES_UPC], 3)
)

print(
    "Hybrid upward price-response violations:",
    hybrid_violations,
)

print(
    "Hybrid predicted-demand range:",
    round(float(np.ptp(hybrid_units)), 2),
)

if any(
    value >= 0
    for value in estimated_elasticities.values()
):
    print(
        "WARNING: Nonnegative elasticity detected. "
        "Pricing model is not approved."
    )
else:
    print(
        "Both fitted elasticities are negative. "
        "Further validation is still required."
    )



PRICEWISE | HYBRID DIAGNOSTICS
Estimated Cheerios elasticity: -1.864
Estimated Corn Flakes elasticity: -2.212
Hybrid upward price-response violations: 0
Hybrid predicted-demand range: 26.38
Both fitted elasticities are negative. Further validation is still required.


In [13]:

# Cell 13 — Diagnose hybrid errors

audit = val_df[
    [
        "UPC", "STORE", "WEEK",
        "UNIT_PRICE", "LAG_1_PRICE",
        "PROMO_CONTEXT", "MOVE",
    ]
].copy()

# Predictions from our trained models
audit["rf_pred"] = val_predictions
audit["rf_reference_pred"] = reference_predictions
audit["monotonic_pred"] = monotone_predictions
audit["hybrid_pred"] = hybrid_predictions

# Calculate the observed week-over-week price change
audit["price_change_pct"] = 100 * (
    audit["UNIT_PRICE"] / audit["LAG_1_PRICE"] - 1
)

audit["abs_price_change_pct"] = (
    audit["price_change_pct"].abs()
)

# Group observations by magnitude of price change
audit["price_band"] = pd.cut(
    audit["abs_price_change_pct"],
    bins=[0, 5, 10, 25, 50, np.inf],
    labels=[
        "0–5%",
        "5–10%",
        "10–25%",
        "25–50%",
        ">50%",
    ],
    include_lowest=True,
)

models = {
    "Random Forest": "rf_pred",
    "RF Reference": "rf_reference_pred",
    "Monotonic": "monotonic_pred",
    "Hybrid": "hybrid_pred",
}

records = []

for band, subset in audit.groupby(
    "price_band", observed=True
):
    row = {
        "Price change": str(band),
        "Observations": len(subset),
        "Share (%)": 100 * len(subset) / len(audit),
    }

    for name, column in models.items():
        row[f"{name} MAE"] = (
            subset["MOVE"] - subset[column]
        ).abs().mean()

    records.append(row)

error_by_price = pd.DataFrame(records)

display(error_by_price.round(2))


,Price change,Observations,Share (%),Random Forest MAE,RF Reference MAE,Monotonic MAE,Hybrid MAE
0,0–5%,6078,80.21,12.39,11.31,14.00,11.28
1,5–10%,101,1.33,62.76,26.10,56.37,26.12
2,10–25%,131,1.73,71.41,101.62,125.35,168.08
3,25–50%,795,10.49,120.24,164.96,155.93,278.84
4,>50%,473,6.24,42.90,73.61,55.78,50.27


In [14]:

# Cell 14 — Model error by price-change magnitude

mae_columns = [
    "Random Forest MAE",
    "Monotonic MAE",
    "Hybrid MAE",
]

plot_errors = error_by_price.melt(
    id_vars=["Price change", "Observations"],
    value_vars=mae_columns,
    var_name="Model",
    value_name="MAE",
)

fig = px.bar(
    plot_errors,
    x="Price change",
    y="MAE",
    color="Model",
    barmode="group",
    hover_data=["Observations"],
    title="PriceWise | Forecast Error by Price-Change Magnitude",
    labels={
        "Price change": "Absolute Price Change",
        "MAE": "Mean Absolute Error (Units)",
    },
)

fig.update_layout(
    template="plotly_white",
    height=500,
)

fig.show()


In [15]:

# Cell 15 — Inspect extreme hybrid forecasting errors

audit["hybrid_abs_error"] = (
    audit["MOVE"] - audit["hybrid_pred"]
).abs()

audit["rf_abs_error"] = (
    audit["MOVE"] - audit["rf_pred"]
).abs()

print("PRICEWISE | WORST HYBRID FORECASTS")
print("=" * 40)

display(
    audit.nlargest(10, "hybrid_abs_error")[
        [
            "UPC",
            "STORE",
            "WEEK",
            "UNIT_PRICE",
            "LAG_1_PRICE",
            "price_change_pct",
            "PROMO_CONTEXT",
            "MOVE",
            "rf_pred",
            "hybrid_pred",
            "hybrid_abs_error",
        ]
    ].round(2)
)

print("\nPRICE-CHANGE COVERAGE")

for threshold in [5, 10, 25, 50]:
    count = int(
        (audit["abs_price_change_pct"] > threshold).sum()
    )

    print(
        f"Price moves > {threshold}%: "
        f"{count:,} ({100 * count / len(audit):.1f}%)"
    )


PRICEWISE | WORST HYBRID FORECASTS


,UPC,STORE,WEEK,UNIT_PRICE,LAG_1_PRICE,price_change_pct,PROMO_CONTEXT,MOVE,rf_pred,hybrid_pred,hybrid_abs_error
5519,1600066610,48,341,2.0,3.35,-40.30,S,72,79.86,1926.23,1854.23
15536,1600066610,90,341,2.0,3.47,-42.36,S,195,92.38,2035.38,1840.38
1705,1600066610,12,341,2.0,3.47,-42.36,S,157,123.18,1980.07,1823.07
302,1600066610,2,341,2.0,3.47,-42.36,S,213,110.18,2031.30,1818.30
17282,1600066610,95,341,2.0,3.47,-42.36,S,314,108.33,2123.41,1809.41
11569,1600066610,75,341,2.0,3.47,-42.36,S,243,108.48,2010.67,1767.67
21780,1600066610,111,341,2.0,3.47,-42.36,S,147,99.90,1906.71,1759.71
20345,1600066610,106,341,2.0,3.35,-40.30,S,162,75.41,1906.30,1744.30
16255,1600066610,92,341,2.0,3.35,-40.30,S,171,81.19,1907.05,1736.05
25605,1600066610,123,341,2.0,3.47,-42.36,S,316,110.31,2025.79,1709.79



PRICE-CHANGE COVERAGE
Price moves > 5%: 1,500 (19.8%)
Price moves > 10%: 1,399 (18.5%)
Price moves > 25%: 1,268 (16.7%)
Price moves > 50%: 473 (6.2%)


In [16]:

# Cell 16 — Explain the worst hybrid prediction

worst_index = audit["hybrid_abs_error"].idxmax()
worst = audit.loc[worst_index]

# Recover the original model inputs.
original_input = X_val.loc[[worst_index]].copy()

# Construct the same reference scenario
# used inside hybrid_predict().
reference_input = original_input.copy()

reference_input["CANDIDATE_PRICE"] = (
    reference_input["LAG_1_PRICE"]
)
reference_input["PRICE_CHANGE_PCT"] = 0.0

# Reference with the recorded promotion.
ref_with_promo = float(
    demand_model.predict(reference_input)[0]
)

# Diagnostic only: same reference price,
# but an unmarked promotion context.
unmarked_input = reference_input.copy()
unmarked_input["PROMO_CONTEXT"] = "UNMARKED"

ref_unmarked = float(
    demand_model.predict(unmarked_input)[0]
)

# Reconstruct the elasticity adjustment.
upc = str(worst["UPC"])
epsilon = estimated_elasticities[upc]

factor = (
    worst["UNIT_PRICE"] / worst["LAG_1_PRICE"]
) ** epsilon

print("PRICEWISE | EXTREME FORECAST INVESTIGATION")
print("=" * 46)

print("Week:", int(worst["WEEK"]))
print("Store:", int(worst["STORE"]))
print("Promotion:", worst["PROMO_CONTEXT"])
print("Actual units:", worst["MOVE"])

print("\nReference prediction with promotion:",
      round(ref_with_promo, 2))

print("Reference prediction unmarked:",
      round(ref_unmarked, 2))

print("Elasticity multiplier:", round(factor, 3))

print("Reconstructed hybrid prediction:",
      round(ref_with_promo * factor, 2))

print("Original hybrid prediction:",
      round(worst["hybrid_pred"], 2))

assert np.isclose(
    ref_with_promo * factor,
    worst["hybrid_pred"],
    rtol=1e-6,
)


PRICEWISE | EXTREME FORECAST INVESTIGATION
Week: 341
Store: 48
Promotion: S
Actual units: 72

Reference prediction with promotion: 736.52
Reference prediction unmarked: 15.52
Elasticity multiplier: 2.615
Reconstructed hybrid prediction: 1926.23
Original hybrid prediction: 1926.23


In [17]:

# Cell 17 — Validate our proposed pricing scope

# Historical support comes from TRAINING ONLY.
train_support = (
    train_df.groupby(
        ["UPC", "STORE"],
        as_index=False
    )
    .agg(
        p10=("UNIT_PRICE", lambda x: x.quantile(0.10)),
        p90=("UNIT_PRICE", lambda x: x.quantile(0.90)),
        training_rows=("MOVE", "size"),
        price_levels=("UNIT_PRICE", "nunique"),
    )
)

scoped = audit.merge(
    train_support,
    on=["UPC", "STORE"],
    how="left",
    validate="many_to_one",
)

# Proposed operational requirements.
eligible = (
    scoped["PROMO_CONTEXT"].eq("UNMARKED")
    & scoped["abs_price_change_pct"].le(10)
    & scoped["training_rows"].ge(30)
    & scoped["price_levels"].ge(5)
    & scoped["UNIT_PRICE"].ge(scoped["p10"])
    & scoped["UNIT_PRICE"].le(scoped["p90"])
    & scoped["LAG_1_PRICE"].ge(scoped["p10"])
    & scoped["LAG_1_PRICE"].le(scoped["p90"])
)

scoped = scoped.loc[eligible].copy()

print("PRICEWISE | BOUNDED VALIDATION")
print("=" * 40)

print("Validation rows:", len(audit))
print("Eligible rows:", len(scoped))
print(
    "Coverage:",
    round(100 * len(scoped) / len(audit), 2),
    "%"
)

if scoped.empty:
    raise ValueError("No observations passed the scope rules")

results_scoped = []

for name, pred_col in [
    ("Random Forest", "rf_pred"),
    ("Monotonic Boosting", "monotonic_pred"),
    ("Hybrid Demand", "hybrid_pred"),
]:
    result = evaluate(
        name,
        scoped["MOVE"],
        scoped[pred_col],
    )
    results_scoped.append(result)

display(
    pd.DataFrame(results_scoped)
    .sort_values("MAE")
    .round(2)
)

# Check each product independently.
product_scoped_results = []

for upc, group in scoped.groupby("UPC"):
    for name, pred_col in [
        ("Random Forest", "rf_pred"),
        ("Monotonic Boosting", "monotonic_pred"),
        ("Hybrid Demand", "hybrid_pred"),
    ]:
        result = evaluate(
            name,
            group["MOVE"],
            group[pred_col],
        )

        product_scoped_results.append({
            "Product": product_names[str(upc)],
            "Observations": len(group),
            **result,
        })

display(
    pd.DataFrame(product_scoped_results)
    .round(2)
)


PRICEWISE | BOUNDED VALIDATION
Validation rows: 7578
Eligible rows: 2583
Coverage: 34.09 %


,Model,MAE,RMSE,WAPE (%)
2,Hybrid Demand,10.21,15.59,28.60
1,Monotonic Boosting,10.55,17.85,29.56
0,Random Forest,11.21,21.29,31.39


,Product,Observations,Model,MAE,RMSE,WAPE (%)
0,Cheerios 15 oz,2393,Random Forest,11.00,18.54,30.22
1,Cheerios 15 oz,2393,Monotonic Boosting,10.25,16.31,28.15
2,Cheerios 15 oz,2393,Hybrid Demand,10.24,15.29,28.14
3,Corn Flakes 18 oz,190,Random Forest,13.82,42.79,51.32
4,Corn Flakes 18 oz,190,Monotonic Boosting,14.44,31.37,53.62
5,Corn Flakes 18 oz,190,Hybrid Demand,9.82,18.93,36.48


In [18]:

# Cell 19 — Export Random Forest to ONNX

from pathlib import Path

import onnx
from skl2onnx import convert_sklearn
from skl2onnx.common.data_types import (
    FloatTensorType,
    StringTensorType,
)

MODEL_DIR = ROOT / "models"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

CANDIDATE_PATH = (
    MODEL_DIR / "pricewise_rf.candidate.onnx"
)

FINAL_MODEL_PATH = (
    MODEL_DIR / "pricewise_rf.onnx"
)

# These columns are categorical strings.
CATEGORICAL = [
    "UPC",
    "STORE",
    "PROMO_CONTEXT",
]

# All remaining features are numeric.
NUMERIC = [
    column
    for column in FEATURE_COLUMNS
    if column not in CATEGORICAL
]

# Define the ONNX input contract.
initial_types = []

for column in FEATURE_COLUMNS:
    if column in CATEGORICAL:
        input_type = StringTensorType([None, 1])
    else:
        input_type = FloatTensorType([None, 1])

    initial_types.append((column, input_type))

print("PRICEWISE | ONNX INPUT CONTRACT")

for name, input_type in initial_types:
    print(name, input_type)

# Convert the complete fitted pipeline,
# including preprocessing and Random Forest.
onnx_model = convert_sklearn(
    demand_model,
    name="PriceWiseDemandRF",
    initial_types=initial_types,
    target_opset=17,
)

# Check ONNX graph validity.
onnx.checker.check_model(onnx_model)

# Save as a candidate, not yet approved.
onnx.save(onnx_model, CANDIDATE_PATH)

print("\nONNX conversion completed")
print("Candidate artifact:", CANDIDATE_PATH)
print(
    "Size:",
    round(CANDIDATE_PATH.stat().st_size / 1024**2, 2),
    "MB",
)


PRICEWISE | ONNX INPUT CONTRACT
UPC StringTensorType(shape=[None, 1])
STORE StringTensorType(shape=[None, 1])
CANDIDATE_PRICE FloatTensorType(shape=[None, 1])
LAG_1_PRICE FloatTensorType(shape=[None, 1])
LAG_1_UNITS FloatTensorType(shape=[None, 1])
AVG_UNITS_PREV_4W FloatTensorType(shape=[None, 1])
PRICE_CHANGE_PCT FloatTensorType(shape=[None, 1])
WEEK_SIN FloatTensorType(shape=[None, 1])
WEEK_COS FloatTensorType(shape=[None, 1])
PROMO_CONTEXT StringTensorType(shape=[None, 1])

ONNX conversion completed
Candidate artifact: /home/amir/Projects/Full/PriceWise_Starter/models/pricewise_rf.candidate.onnx
Size: 5.74 MB


In [19]:

# Cell 20 — ONNX Runtime prediction parity

import numpy as np
import onnxruntime as ort

session = ort.InferenceSession(
    str(CANDIDATE_PATH),
    providers=["CPUExecutionProvider"],
)


def onnx_predict(frame: pd.DataFrame) -> np.ndarray:
    """Run the converted Random Forest pipeline."""

    feeds = {}

    for model_input in session.get_inputs():
        name = model_input.name

        if name in CATEGORICAL:
            values = frame[name].astype(str).to_numpy()
        else:
            values = frame[name].to_numpy(
                dtype=np.float32
            )

        feeds[name] = values.reshape(-1, 1)

    output = session.run(None, feeds)[0]

    return np.asarray(output).reshape(-1)


# Ensure the graph has our expected inputs.
actual_inputs = {
    value.name for value in session.get_inputs()
}

assert actual_inputs == set(FEATURE_COLUMNS)

# Predictions from the original model.
native_predictions = np.maximum(
    demand_model.predict(X_val),
    0,
)

# Predictions from ONNX Runtime.
runtime_predictions = np.maximum(
    onnx_predict(X_val),
    0,
)

# Numerical comparison.
absolute_difference = np.abs(
    native_predictions - runtime_predictions
)

print("PRICEWISE | INFERENCE PARITY")
print("=" * 40)

print("Predictions checked:", len(X_val))

print(
    "Mean absolute difference:",
    round(float(absolute_difference.mean()), 6),
)

print(
    "99th percentile difference:",
    round(float(np.quantile(
        absolute_difference, 0.99
    )), 6),
)

print(
    "Maximum difference:",
    round(float(absolute_difference.max()), 6),
)

# Initial numerical parity requirement.
# A small absolute/relative difference is expected
# because ONNX may use float32 arithmetic.
np.testing.assert_allclose(
    runtime_predictions,
    native_predictions,
    rtol=0.02,
    atol=0.5,
)

print("\nPASS: Native and ONNX predictions agree")


PRICEWISE | INFERENCE PARITY
Predictions checked: 7578
Mean absolute difference: 2e-06
99th percentile difference: 2e-05
Maximum difference: 9.4e-05

PASS: Native and ONNX predictions agree


In [20]:

# Cell 21 — Hybrid prediction with either inference engine

def hybrid_with_engine(frame, predict_reference):
    """Price-adjust reference demand using elasticities."""

    X = frame.copy()

    candidate = X["CANDIDATE_PRICE"].to_numpy(
        dtype=float
    )

    previous = X["LAG_1_PRICE"].to_numpy(
        dtype=float
    )

    if (
        not np.isfinite(candidate).all()
        or not np.isfinite(previous).all()
        or (candidate <= 0).any()
        or (previous <= 0).any()
    ):
        raise ValueError("Invalid candidate/reference prices")

    epsilon = (
        X["UPC"]
        .astype(str)
        .map(estimated_elasticities)
        .to_numpy(dtype=float)
    )

    if not np.isfinite(epsilon).all():
        raise ValueError("Unsupported product elasticity")

    # Construct reference-price features.
    reference = X.copy()

    reference["CANDIDATE_PRICE"] = (
        reference["LAG_1_PRICE"]
    )

    reference["PRICE_CHANGE_PCT"] = 0.0

    reference_demand = np.maximum(
        predict_reference(reference),
        0,
    )

    return (
        reference_demand
        * np.power(candidate / previous, epsilon)
    )


# Test identical input rows through both engines.
sample = X_val.sample(
    n=min(500, len(X_val)),
    random_state=42,
)

native_hybrid = hybrid_with_engine(
    sample,
    demand_model.predict,
)

onnx_hybrid = hybrid_with_engine(
    sample,
    onnx_predict,
)

difference = np.abs(
    native_hybrid - onnx_hybrid
)

print("PRICEWISE | HYBRID INFERENCE PARITY")
print("=" * 40)

print("Scenarios checked:", len(sample))
print(
    "Max absolute difference:",
    round(float(difference.max()), 6)
)

np.testing.assert_allclose(
    onnx_hybrid,
    native_hybrid,
    rtol=0.02,
    atol=1.0,
)

print("\nPASS: Hybrid works with ONNX inference")


PRICEWISE | HYBRID INFERENCE PARITY
Scenarios checked: 500
Max absolute difference: 0.00017

PASS: Hybrid works with ONNX inference


In [21]:

# Cell 22 — Save deployment metadata

import json

# Product-store support calculated from training only.
support_records = []

for row in train_support.itertuples(index=False):
    support_records.append({
        "upc": str(row.UPC),
        "store": int(row.STORE),
        "p10": float(row.p10),
        "p90": float(row.p90),
        "training_rows": int(row.training_rows),
        "price_levels": int(row.price_levels),
    })

metadata = {
    "model_name": "PriceWise Demand Engine",
    "version": "0.1.0",
    "training_last_week": 257,
    "elasticities": {
        upc: float(value)
        for upc, value in estimated_elasticities.items()
    },
    "pricing_policy": {
        "supported_promotion": "UNMARKED",
        "max_abs_price_change_pct": 10.0,
        "min_training_rows": 30,
        "min_price_levels": 5,
        "historical_price_quantiles": [0.10, 0.90],
    },
    "product_store_support": support_records,
    "validation": {
        "bounded_rows": 2583,
        "bounded_coverage_pct": 34.09,
        "bounded_hybrid_mae": 10.21,
        "onnx_parity_max_abs_difference": float(
            absolute_difference.max()
        ),
    },
    "limitations": [
        "Historical observational pricing data",
        "Positive-price, positive-sales training subset",
        "Elasticities are not causal estimates",
        "Unmarked promotion does not prove no promotion",
        "Pricing scenarios require validation guardrails",
    ],
}

# Promote the verified candidate model.
CANDIDATE_PATH.replace(FINAL_MODEL_PATH)

METADATA_PATH = MODEL_DIR / "pricewise_policy.json"

METADATA_PATH.write_text(
    json.dumps(metadata, indent=2),
    encoding="utf-8",
)

print("PRICEWISE | MODEL ARTIFACTS SAVED")
print("=" * 40)
print("ONNX:", FINAL_MODEL_PATH)
print("Policy:", METADATA_PATH)
print("Product-store entries:", len(support_records))


PRICEWISE | MODEL ARTIFACTS SAVED
ONNX: /home/amir/Projects/Full/PriceWise_Starter/models/pricewise_rf.onnx
Policy: /home/amir/Projects/Full/PriceWise_Starter/models/pricewise_policy.json
Product-store entries: 172


In [22]:

# Cell 23 — Frozen-model final test

import json
import onnxruntime as ort
import numpy as np
import pandas as pd

FINAL_ONNX = ROOT / "models/pricewise_rf.onnx"
POLICY_PATH = ROOT / "models/pricewise_policy.json"

policy = json.loads(
    POLICY_PATH.read_text(encoding="utf-8")
)

assert policy["training_last_week"] == 257
assert test_df["WEEK"].min() >= 343

# Load the actual deployment artifact.
session = ort.InferenceSession(
    str(FINAL_ONNX),
    providers=["CPUExecutionProvider"],
)

# onnx_predict() from Cell 20 reads this session.


def frozen_hybrid_predict(frame):
    """Hybrid demand using frozen ONNX and saved policy."""

    candidate = frame["CANDIDATE_PRICE"].to_numpy(
        dtype=float
    )
    previous = frame["LAG_1_PRICE"].to_numpy(
        dtype=float
    )

    if (
        not np.isfinite(candidate).all()
        or not np.isfinite(previous).all()
        or (candidate <= 0).any()
        or (previous <= 0).any()
    ):
        raise ValueError("Invalid prices")

    epsilon = (
        frame["UPC"]
        .astype(str)
        .map(policy["elasticities"])
        .to_numpy(dtype=float)
    )

    if not np.isfinite(epsilon).all():
        raise ValueError("Unsupported product")

    # Reference-price scenario.
    reference = frame.copy()
    reference["CANDIDATE_PRICE"] = reference["LAG_1_PRICE"]
    reference["PRICE_CHANGE_PCT"] = 0.0

    baseline = np.maximum(
        onnx_predict(reference),
        0,
    )

    return baseline * np.power(
        candidate / previous,
        epsilon,
    )


# The test inputs and observations must align.
assert X_test.index.equals(test_df.index)

# Predict every test row.
test_results = test_df.copy()

test_results["rf_pred"] = np.maximum(
    onnx_predict(X_test), 0
)

test_results["hybrid_pred"] = (
    frozen_hybrid_predict(X_test)
)

test_results["last_week_pred"] = (
    test_results["LAG_1_UNITS"]
)

test_results["rolling_pred"] = (
    test_results["AVG_UNITS_PREV_4W"]
)

# Compare all methods on the same test rows.
test_models = {
    "Random Forest ONNX": "rf_pred",
    "Frozen Hybrid": "hybrid_pred",
    "Last Week": "last_week_pred",
    "4-Week Average": "rolling_pred",
}

test_metrics = [
    evaluate(
        name,
        test_results["MOVE"],
        test_results[column],
    )
    for name, column in test_models.items()
]

print("PRICEWISE | FROZEN MODEL FINAL TEST")
print("=" * 45)
print("Test rows:", len(test_results))

display(
    pd.DataFrame(test_metrics).round(2)
)


PRICEWISE | FROZEN MODEL FINAL TEST
Test rows: 8157


,Model,MAE,RMSE,WAPE (%)
0,Random Forest ONNX,46.68,182.63,53.22
1,Frozen Hybrid,66.42,250.96,75.73
2,Last Week,121.85,343.77,138.91
3,4-Week Average,113.37,269.95,129.25


In [23]:

# Cell 24 — Frozen operational scope evaluation

support_table = pd.DataFrame(
    policy["product_store_support"]
).rename(columns={
    "upc": "UPC",
    "store": "STORE",
})

support_table["UPC"] = (
    support_table["UPC"].astype(str)
)
support_table["STORE"] = (
    support_table["STORE"].astype(int)
)

assert not support_table.duplicated(
    ["UPC", "STORE"]
).any()

diagnostic = test_results.copy()

diagnostic["UPC"] = (
    diagnostic["UPC"].astype(str)
)

diagnostic = diagnostic.merge(
    support_table,
    on=["UPC", "STORE"],
    how="left",
    validate="many_to_one",
)

rules = policy["pricing_policy"]

price_change = (
    100 * (
        diagnostic["UNIT_PRICE"]
        / diagnostic["LAG_1_PRICE"]
        - 1
    )
).abs()

# Frozen business constraints.
supported = (
    diagnostic["PROMO_CONTEXT"].eq(
        rules["supported_promotion"]
    )
    & price_change.le(
        rules["max_abs_price_change_pct"]
    )
    & diagnostic["training_rows"].ge(
        rules["min_training_rows"]
    )
    & diagnostic["price_levels"].ge(
        rules["min_price_levels"]
    )
    & diagnostic["UNIT_PRICE"].between(
        diagnostic["p10"],
        diagnostic["p90"],
    )
    & diagnostic["LAG_1_PRICE"].between(
        diagnostic["p10"],
        diagnostic["p90"],
    )
)

diagnostic["SUPPORTED"] = (
    supported.fillna(False).astype(bool)
)

scoped_test = diagnostic.loc[
    diagnostic["SUPPORTED"]
].copy()

print("PRICEWISE | FROZEN SCOPE TEST")
print("=" * 45)

print("Total test rows:", len(diagnostic))
print("Supported rows:", len(scoped_test))

print(
    "Coverage:",
    f"{100 * len(scoped_test) / len(diagnostic):.2f}%"
)

if scoped_test.empty:
    print("No supported observations: policy requires review.")
else:
    scoped_metrics = [
        evaluate(
            name,
            scoped_test["MOVE"],
            scoped_test[column],
        )
        for name, column in test_models.items()
    ]

    display(
        pd.DataFrame(scoped_metrics).round(2)
    )

# Product-level support and forecasting performance.
product_names = {
    "1600066610": "Cheerios 15 oz",
    "3800000120": "Corn Flakes 18 oz",
}

reports = []

for upc, product in product_names.items():
    product_all = diagnostic.loc[
        diagnostic["UPC"] == upc
    ]

    product_scoped = product_all.loc[
        product_all["SUPPORTED"]
    ]

    report = {
        "Product": product,
        "Test rows": len(product_all),
        "Supported rows": len(product_scoped),
        "Coverage (%)": (
            100 * len(product_scoped) / len(product_all)
        ),
    }

    if not product_scoped.empty:
        report["RF MAE"] = evaluate(
            "RF",
            product_scoped["MOVE"],
            product_scoped["rf_pred"],
        )["MAE"]

        report["Hybrid MAE"] = evaluate(
            "Hybrid",
            product_scoped["MOVE"],
            product_scoped["hybrid_pred"],
        )["MAE"]

    reports.append(report)

product_test_report = pd.DataFrame(reports)

display(product_test_report.round(2))


PRICEWISE | FROZEN SCOPE TEST
Total test rows: 8157
Supported rows: 2295
Coverage: 28.14%


,Model,MAE,RMSE,WAPE (%)
0,Random Forest ONNX,21.98,112.00,44.82
1,Frozen Hybrid,21.23,112.28,43.27
2,Last Week,31.56,155.46,64.33
3,4-Week Average,61.60,166.37,125.57


,Product,Test rows,Supported rows,Coverage (%),RF MAE,Hybrid MAE
0,Cheerios 15 oz,4087,2252,55.10,22.07,21.40
1,Corn Flakes 18 oz,4070,43,1.06,17.32,11.99


In [24]:

# Cell 25 — Cold-start store diagnostics

training_stores = set(
    train_df["STORE"].astype(int)
)

diagnostic["STORE_GROUP"] = np.where(
    diagnostic["STORE"].isin(training_stores),
    "Seen in training",
    "Unseen in training",
)

cold_start_results = []

for group_name, subset in diagnostic.groupby(
    "STORE_GROUP"
):
    metrics = evaluate(
        group_name,
        subset["MOVE"],
        subset["rf_pred"],
    )

    cold_start_results.append({
        "Store group": group_name,
        "Rows": len(subset),
        "Supported rows": int(
            subset["SUPPORTED"].sum()
        ),
        **{
            key: value
            for key, value in metrics.items()
            if key != "Model"
        },
    })

display(
    pd.DataFrame(cold_start_results).round(2)
)

# Visualize pricing-policy coverage.
coverage_chart = product_test_report.copy()

fig = px.bar(
    coverage_chart,
    x="Product",
    y="Coverage (%)",
    text="Coverage (%)",
    title="PriceWise | Test-Period Pricing-Scope Coverage",
)

fig.update_traces(
    texttemplate="%{text:.1f}%",
    textposition="outside",
)

fig.update_layout(
    template="plotly_white",
    height=430,
    yaxis_range=[0, 105],
)

fig.show()


,Store group,Rows,Supported rows,MAE,RMSE,WAPE (%)
0,Seen in training,7781,2295,47.15,184.03,53.28
1,Unseen in training,376,0,36.95,150.66,51.69


In [25]:

# PriceWise — Profit scenario visualization

from pricewise.pricing.engine import PricingEngine
from pricewise.pricing.policy import PricingPolicy
from pricewise.ml.inference import OnnxDemandPredictor

from plotly.subplots import make_subplots
import plotly.graph_objects as go

engine = PricingEngine(
    policy=PricingPolicy(
        ROOT / "models/pricewise_policy.json"
    ),
    predictor=OnnxDemandPredictor(
        ROOT / "models/pricewise_rf.onnx"
    ),
)

simulation = engine.evaluate(
    sales,
    upc="1600066610",
    store=101,
    forecast_week=400,
)

scenarios = simulation.scenarios

fig = make_subplots(
    rows=2,
    cols=1,
    shared_xaxes=True,
    subplot_titles=[
        "Predicted Weekly Demand",
        "Modeled Weekly Gross Profit",
    ],
    vertical_spacing=0.13,
)

fig.add_trace(
    go.Scatter(
        x=scenarios["price"],
        y=scenarios["predicted_units"],
        mode="lines",
        name="Predicted units",
    ),
    row=1, col=1,
)

fig.add_trace(
    go.Scatter(
        x=scenarios["price"],
        y=scenarios["gross_profit"],
        mode="lines",
        name="Gross profit",
    ),
    row=2, col=1,
)

fig.add_vline(
    x=simulation.reference_price,
    line_dash="dash",
    line_color="gray",
)

fig.update_yaxes(
    title_text="Weekly Units",
    row=1, col=1,
)

fig.update_yaxes(
    title_text="Gross Profit ($)",
    row=2, col=1,
)

fig.update_xaxes(
    title_text="Candidate Price ($)",
    row=2, col=1,
)

fig.update_layout(
    title="PriceWise | Cheerios · Store 101 · Week 400",
    height=700,
    template="plotly_white",
    hovermode="x unified",
)

fig.show()


In [26]:

# Visualize the chosen PriceWise pricing strategies.
# Uses `simulation` from the preceding profit-curve cell.

import plotly.graph_objects as go

from pricewise.pricing.strategies import select_strategies

decisions = select_strategies(simulation)

fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=simulation.scenarios["price"],
        y=simulation.scenarios["gross_profit"],
        mode="lines",
        name="Modeled Gross Profit",
    )
)

colors = {
    "Conservative": "#22916A",
    "Balanced": "#5478E8",
    "Aggressive": "#D98C35",
}

for name, decision in decisions.items():
    if decision.status != "scenario":
        continue

    fig.add_trace(
        go.Scatter(
            x=[decision.price],
            y=[decision.gross_profit],
            mode="markers",
            name=f"{name}: ${decision.price:.2f}",
            marker={
                "size": 15,
                "color": colors[name],
                "symbol": "diamond",
            },
            hovertemplate=(
                f"<b>{name}</b>"
                "<br>Price: $%{x:.2f}"
                "<br>Modeled Profit: $%{y:.2f}"
                "<extra></extra>"
            ),
        )
    )

fig.add_vline(
    x=simulation.reference_price,
    line_dash="dash",
    line_color="gray",
    annotation_text="Reference Price",
)

fig.update_layout(
    title="PriceWise | Conservative vs Balanced vs Aggressive",
    xaxis_title="Candidate Price ($)",
    yaxis_title="Modeled Weekly Gross Profit ($)",
    template="plotly_white",
    hovermode="closest",
    height=550,
)

fig.show()
